# 中际旭创（300308.SZ）股票技术指标分析

**数据范围**: 2023年7月 - 2026年7月（近三年日度交易数据）  
**数据源**: AKShare（前复权数据）✅  
**分析指标**: RSI、MACD、布林带、KDJ  
**目标**: 通过技术指标计算中际旭创的买卖信号和趋势判断

---

## 📊 数据说明

**✅ 已使用前复权数据**  
- 中际旭创在近三年内存在多次除权除息（2024-06-06: 10转4派4.5元）  
- 使用前复权数据消除价格缺口，确保技术指标计算准确  
- 验证：2024-06-06 单日价格变化从 -24.88%（不复权）调整为 +4.91%（前复权）✅


## 第一部分：环境准备与数据加载


In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.font_manager import FontProperties
import warnings
warnings.filterwarnings('ignore')

# ✅ 中文字体配置（最可靠的方法）
chinese_font = FontProperties(fname=r'C:\Windows\Fonts\msyh.ttc', size=12)
chinese_font_title = FontProperties(fname=r'C:\Windows\Fonts\msyh.ttc', size=16)
chinese_font_legend = FontProperties(fname=r'C:\Windows\Fonts\msyh.ttc', size=11)
chinese_font_label = FontProperties(fname=r'C:\Windows\Fonts\msyh.ttc', size=12)

# 设置图表风格（白色背景，浅色细网格）
plt.style.use('default')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['grid.linestyle'] = '-'
plt.rcParams['grid.linewidth'] = 0.5
plt.rcParams['axes.unicode_minus'] = False

print('✅ 所有库已成功导入')
print('✅ 中文字体已配置: Microsoft YaHei')
print('✅ 图表风格已设置: 白色背景 + 浅色细网格')


In [ ]:
# 加载数据
file_path = r'E:\BA课程练习\task2\中际旭创_日度交易数据.csv'
df = pd.read_csv(file_path)

# 数据预处理
df['trade_date'] = pd.to_datetime(df['trade_date'], format='%Y%m%d')
df = df.sort_values('trade_date').reset_index(drop=True)

print('✅ 数据加载成功！')
print(f'数据形状: {df.shape}')
print(f'时间范围: {df["trade_date"].min()} 至 {df["trade_date"].max()}')
print(f'交易日数量: {len(df)}')
df.head()


## 第二部分：数据诊断分析


In [ ]:
# 检查缺失值
missing_values = df.isnull().sum()
if missing_values.sum() == 0:
    print('✅ 数据无缺失值！')
else:
    print(missing_values[missing_values > 0])

# 描述性统计量
print('\n=== 描述性统计量 ===')
display(df[['open', 'high', 'low', 'close']].describe())


## 第三部分：RSI (相对强弱指标)

### 理论介绍

**定义**: RSI 衡量股价涨跌速度，取值范围 0~100。  
**判断标准**: RSI > 70 超买，RSI < 30 超卖。


In [ ]:
# RSI 计算
def calculate_rsi(data, period=14):
    delta = data.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.ewm(com=period-1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period-1, min_periods=period).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df['RSI_14'] = calculate_rsi(df['close'])
print('✅ RSI 指标计算完成！')
df[['trade_date', 'close', 'RSI_14']].tail(10)


In [ ]:
# RSI 可视化
fig, ax = plt.subplots(figsize=(14, 6))

# 绘制 RSI 曲线（线条变细）
ax.plot(df['trade_date'], df['RSI_14'], 
        color='blue', linewidth=1.0, label='RSI (14)')

# 添加超买线和超卖线
ax.axhline(y=70, color='red', linestyle='--', linewidth=0.8, alpha=0.7, label='超买线 (70)')
ax.axhline(y=30, color='green', linestyle='--', linewidth=0.8, alpha=0.7, label='超卖线 (30)')
ax.axhline(y=50, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)

# 填充超买超卖区域
ax.fill_between(df['trade_date'], 70, 100, alpha=0.1, color='red')
ax.fill_between(df['trade_date'], 0, 30, alpha=0.1, color='green')

# 设置标题和标签（使用 fontproperties）
ax.set_title('中际旭创 RSI 指标 (14天)', fontproperties=chinese_font_title, fontweight='bold')
ax.set_xlabel('日期', fontproperties=chinese_font_label)
ax.set_ylabel('RSI 值', fontproperties=chinese_font_label)

# ✅ 图例位置改为右上
ax.legend(loc='upper right', prop=chinese_font_legend)

# 格式化X轴日期
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


## 第四部分：MACD (移动平均收敛/发散指标)

### 理论介绍

**定义**: MACD 基于移动平均线的交叉原理，捕捉趋势反转点。  
**判断标准**: DIF 上穿 DEA 为金叉（买入），DIF 下穿 DEA 为死叉（卖出）。


In [ ]:
# MACD 计算
def calculate_macd(data, fast=12, slow=26, signal=9):
    ema_fast = data.ewm(span=fast, adjust=False).mean()
    ema_slow = data.ewm(span=slow, adjust=False).mean()
    dif = ema_fast - ema_slow
    dea = dif.ewm(span=signal, adjust=False).mean()
    macd = (dif - dea) * 2
    return dif, dea, macd

df['DIF'], df['DEA'], df['MACD'] = calculate_macd(df['close'])
print('✅ MACD 指标计算完成！')
df[['trade_date', 'close', 'DIF', 'DEA', 'MACD']].tail(10)


In [ ]:
# MACD 可视化（线条变细）
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# 子图1：DIF 和 DEA
axes[0].plot(df['trade_date'], df['DIF'], 
            color='blue', linewidth=1.0, label='DIF (快线)')
axes[0].plot(df['trade_date'], df['DEA'], 
            color='red', linewidth=1.0, label='DEA (慢线)')
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
axes[0].set_title('MACD: DIF 与 DEA', fontproperties=chinese_font_title, fontweight='bold')
axes[0].set_ylabel('数值', fontproperties=chinese_font_label)
axes[0].legend(loc='upper right', prop=chinese_font_legend)
axes[0].grid(True, alpha=0.3)

# 子图2：MACD 柱状图
colors = ['red' if x > 0 else 'green' for x in df['MACD']]
axes[1].bar(df['trade_date'], df['MACD'], 
            color=colors, alpha=0.7, width=0.8, label='MACD 柱')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
axes[1].set_title('MACD 柱状图', fontproperties=chinese_font_title, fontweight='bold')
axes[1].set_xlabel('日期', fontproperties=chinese_font_label)
axes[1].set_ylabel('MACD 值', fontproperties=chinese_font_label)
axes[1].legend(loc='upper right', prop=chinese_font_legend)
axes[1].grid(True, alpha=0.3)

# 格式化X轴日期
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


## 第五部分：布林带 (Bollinger Bands)

### 理论介绍

**定义**: 布林带由三条轨道线组成：中轨（20日MA）、上轨（中轨+2倍标准差）、下轨（中轨-2倍标准差）。  
**判断标准**: 价格触及上轨可能回调，触及下轨可能反弹。


In [ ]:
# 布林带计算 + 可视化（合并单元格，确保列一定存在）

# ✅ 步骤1：计算布林带
print('=== 布林带计算 ===')

def calculate_bollinger_bands(data, period=20, std_dev=2):
    middle_band = data.rolling(window=period).mean()
    std = data.rolling(window=period).std()
    upper_band = middle_band + (std * std_dev)
    lower_band = middle_band - (std * std_dev)
    return middle_band, upper_band, lower_band

df['BB_middle'], df['BB_upper'], df['BB_lower'] = calculate_bollinger_bands(df['close'])
df['BB_width'] = df['BB_upper'] - df['BB_lower']

print('✅ 布林带计算完成！')
print(f'当前收盘价: {df["close"].iloc[-1]:.2f} 元')
print(f'当前上轨: {df["BB_upper"].iloc[-1]:.2f} 元')
print(f'当前下轨: {df["BB_lower"].iloc[-1]:.2f} 元')

# ✅ 步骤2：可视化
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# 子图1：股价 + 布林带
axes[0].plot(df['trade_date'], df['close'], 
            color='black', linewidth=1.0, label='收盘价')
axes[0].plot(df['trade_date'], df['BB_middle'], 
            color='blue', linewidth=0.8, linestyle='--', label='中轨')
axes[0].plot(df['trade_date'], df['BB_upper'], 
            color='red', linewidth=0.8, linestyle='--', label='上轨')
axes[0].plot(df['trade_date'], df['BB_lower'], 
            color='green', linewidth=0.8, linestyle='--', label='下轨')

# 填充上下轨区间
axes[0].fill_between(df['trade_date'], df['BB_upper'], df['BB_lower'], 
                       alpha=0.1, color='gray')

axes[0].set_title('中际旭创 布林带', fontproperties=chinese_font_title, fontweight='bold')
axes[0].set_ylabel('价格 (元)', fontproperties=chinese_font_label)
axes[0].legend(loc='upper left', prop=chinese_font_legend)
axes[0].grid(True, alpha=0.3)

# 子图2：带宽
axes[1].plot(df['trade_date'], df['BB_width'], 
             color='purple', linewidth=1.0, label='布林带宽')
axes[1].set_title('布林带宽 (波动率指标)', fontproperties=chinese_font_title, fontweight='bold')
axes[1].set_xlabel('日期', fontproperties=chinese_font_label)
axes[1].set_ylabel('带宽', fontproperties=chinese_font_label)
axes[1].legend(loc='upper right', prop=chinese_font_legend)
axes[1].grid(True, alpha=0.3)

# 格式化X轴日期
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


## 第六部分：KDJ 指标

### 理论介绍

**定义**: KDJ 是基于RSV的随机指标，由K线、D线、J线组成。  
**重要特性**: J值可以为负数或大于100。  
**判断标准**: K > D 为多头，K < D 为空头。


In [ ]:
# KDJ 计算
def calculate_kdj(high, low, close, period=9):
    low_list = low.rolling(window=period).min()
    high_list = high.rolling(window=period).max()
    rsv = (close - low_list) / (high_list - low_list) * 100
    K = rsv.ewm(com=2, adjust=False).mean()
    D = K.ewm(com=2, adjust=False).mean()
    J = 3 * K - 2 * D
    return K, D, J

df['K'], df['D'], df['J'] = calculate_kdj(df['high'], df['low'], df['close'])
print('✅ KDJ 指标计算完成！')
print(f'K 范围: {df["K"].min():.2f} ~ {df["K"].max():.2f}')
print(f'D 范围: {df["D"].min():.2f} ~ {df["D"].max():.2f}')
print(f'J 范围: {df["J"].min():.2f} ~ {df["J"].max():.2f}')
df[['trade_date', 'close', 'K', 'D', 'J']].tail(10)


In [ ]:
# KDJ 可视化（黑金紫配色，线条变细）
fig, ax = plt.subplots(figsize=(14, 6))

# ✅ 绘制 K、D、J 曲线（黑金紫配色）
ax.plot(df['trade_date'], df['K'], 
        color='black', linewidth=1.0, label='K 线')
ax.plot(df['trade_date'], df['D'], 
        color='#DAA520', linewidth=1.0, label='D 线 (金色)')
ax.plot(df['trade_date'], df['J'], 
        color='purple', linewidth=0.8, label='J 线')

# 添加超买线和超卖线
ax.axhline(y=80, color='red', linestyle='--', linewidth=0.8, alpha=0.7, label='超买线 (80)')
ax.axhline(y=20, color='green', linestyle='--', linewidth=0.8, alpha=0.7, label='超卖线 (20)')

# ✅ 不限制Y轴范围，让J值自然展示（J可以为负）
y_min, y_max = ax.get_ylim()
ax.fill_between(df['trade_date'], 80, y_max, alpha=0.1, color='red')
ax.fill_between(df['trade_date'], y_min, 20, alpha=0.1, color='green')

# 设置标题和标签
ax.set_title('中际旭创 KDJ 指标', fontproperties=chinese_font_title, fontweight='bold')
ax.set_xlabel('日期', fontproperties=chinese_font_label)
ax.set_ylabel('KDJ 值', fontproperties=chinese_font_label)

# ✅ 图例位置改为左上
ax.legend(loc='upper left', prop=chinese_font_legend)

# 格式化X轴日期
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


## 第七部分：总结

### 已完成的技术指标

1. RSI (相对强弱指标) - 判断超买超卖
2. MACD (移动平均收敛/发散) - 捕捉趋势反转
3. 布林带 (Bollinger Bands) - 衡量价格波动区间
4. KDJ (随机指标) - 敏感捕捉短期转折点

---

**免责声明**: 本分析仅供参考，不构成任何投资建议。股市有风险，投资需谨慎。
